In [1]:
from utils.utils import process_txt_files, plot_categorical_distribution, get_stat_significance_bootstrap
import pandas as pd
import numpy as np
from datasets import load_dataset
import shutup
shutup.please()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vojta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
C:\Users\vojta\miniconda3\envs\llm-features\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
folder_path = "../../data/outputs/hazard"
df = process_txt_files(folder_path, "hazard")
df.shape

(6644, 22)

In [3]:
import requests
import pandas as pd
from io import StringIO

# URLs for the files
urls = [
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_train.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_valid.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_test.csv"
]

# Load each file into a DataFrame
dataframes = []
for url in urls:
    response = requests.get(url)
    response.raise_for_status()  # Raise an error for bad status codes
    csv_data = StringIO(response.text)  # Convert text to a file-like object
    df_orig = pd.read_csv(csv_data)
    dataframes.append(df_orig)


# Access the DataFrames
train_df, valid_df, test_df = dataframes

# Example: Display the first few rows of the training DataFrame
print(train_df.head())

   Unnamed: 0  year  month  day country                             title  \
0           0  1994      1    7      us  Recall Notification: FSIS-024-94   
1           1  1994      3   10      us  Recall Notification: FSIS-033-94   
2           2  1994      3   28      us  Recall Notification: FSIS-014-94   
3           3  1994      4    3      us  Recall Notification: FSIS-009-94   
4           4  1994      7    1      us  Recall Notification: FSIS-001-94   

                                                text hazard-category  \
0  Case Number: 024-94   \n            Date Opene...      biological   
1  Case Number: 033-94   \n            Date Opene...      biological   
2  Case Number: 014-94   \n            Date Opene...      biological   
3  Case Number: 009-94   \n            Date Opene...  foreign bodies   
4  Case Number: 001-94   \n            Date Opene...  foreign bodies   

               product-category                  hazard  \
0  meat, egg and dairy products  listeria mon

In [4]:
df_comb = pd.concat([train_df, valid_df, test_df], axis=0, ignore_index=True)
df['hazard-category'] = df_comb['hazard-category']

In [6]:
df.columns

Index(['id', 'custom_id', 'product_type', 'hazard_type', 'recall_reason',
       'contaminant_type', 'distribution_area', 'recall_action',
       'affected_population', 'recall_date', 'company_name',
       'product_packaging', 'allergen_type', 'recall_severity',
       'product_origin', 'consumer_advice', 'recall_source',
       'product_batch_code', 'recall_notification_method', 'product_size',
       'recall_frequency', 'recall_duration', 'hazard-category'],
      dtype='object')

In [7]:
df.drop(columns = ['Unnamed: 0'], inplace = True, errors = 'ignore')

In [8]:
df.drop(columns = ['custom_id', 'id'], inplace = True, errors = 'ignore')
for col in df.columns:
    df[col] = df[col].apply(lambda x: str(x) if isinstance(x, list) else x)

In [14]:
df.head()

,product_type,hazard_type,recall_reason,contaminant_type,distribution_area,recall_action,affected_population,recall_date,company_name,product_packaging,...,recall_severity,product_origin,consumer_advice,recall_source,product_batch_code,recall_notification_method,product_size,recall_frequency,recall_duration,hazard-category
0,Chicken,biological,Contamination,Listeria,National,Product Recall,General Public,1994-07-01,GERHARD'S NAPA VALLEY SAUSAGE,Other,...,Class 1,USA,Do Not Consume,USDA,024-94,Press Release,"2,894",Low,Medium,biological
1,Other,biological,Contamination,Listeria,National,Product Recall,General Public,1994-10-03,WIMMER'S MEAT PRODUCTS,Other,...,Class 1,USA,Do Not Consume,USDA,N/A,Press Release,"5,500",Low,Long,biological
2,Other,biological,Contamination,Listeria,National,Product Recall,General Public,1994-03-28,WILLOW FOODS INC,Other,...,Class 1,USA,Do Not Consume,USDA,014-94,Press Release,"3,950",Low,Medium,biological
3,Other,foreign bodies,Contamination,Plastic,National,Product Recall,General Public,1994-03-04,OSCAR MAYER FOODS,Other,...,Class 2,USA,Do Not Consume,USDA,009-94,Press Release,"16,164 pounds",Low,Medium,foreign bodies
4,Chicken,foreign bodies,Contamination,Plastic,National,Product Recall,General Public,1994-01-07,TYSON FOODS,Plastic,...,Class 1,USA,Do Not Consume,USDA,001-94,Press Release,"46,168 pounds",Low,Medium,foreign bodies


In [17]:
for col in df.drop(columns = ['hazard-category']).columns:
    get_stat_significance_bootstrap(df, col, 'hazard-category', n_bootstrap=2500)


Original P-value: 9.808987707681857e-281
Bootstrap Mean P-value: 1.1241195819063386e-219
Relationship is significant in 2500/2500 bootstrap samples.
Cramér's V (Effect Size): 0.1636166414071289
Effect Size: Moderate association
Significant association between product_type and hazard-category.
---------------------------------------------------------------
Original P-value: 0.0
Bootstrap Mean P-value: 0.0
Relationship is significant in 2500/2500 bootstrap samples.
Cramér's V (Effect Size): 0.8170584729957818
Effect Size: Strong association
Significant association between hazard_type and hazard-category.
---------------------------------------------------------------
Original P-value: 0.0
Bootstrap Mean P-value: 0.0
Relationship is significant in 2500/2500 bootstrap samples.
Cramér's V (Effect Size): 0.5818949578326555
Effect Size: Strong association
Significant association between recall_reason and hazard-category.
---------------------------------------------------------------
Original